In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import os
import sys
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error
import time
import warnings
warnings.filterwarnings('ignore')

# Windows VS Code 터미널 호환: 진행바 한 줄 덮어쓰기
def _print_progress(msg):
    sys.stdout.write('\r' + msg)
    sys.stdout.flush()

# ============================================================
# 실시간 출력 헬퍼
# ============================================================
def elapsed(start):
    s = int(time.time() - start)
    return f"{s//60}m {s%60:02d}s"

def section(title):
    print(f"\n{'='*55}\n  {title}\n{'='*55}")

class LGBProgressCallback:
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold = fold; self.n_folds = n_folds
        self.total = total_rounds; self.log_every = log_every
        self.start = time.time(); self.best_mae = float('inf'); self.best_iter = 0
    def __call__(self, env):
        it  = env.iteration + 1
        mae = env.evaluation_result_list[0][2]
        if mae < self.best_mae: self.best_mae = mae; self.best_iter = it
        if it % self.log_every == 0 or it == self.total:
            filled = int(30 * it / self.total)
            bar = '█'*filled + '░'*(30-filled)
            _print_progress(
                f"  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%"
                f"  iter {it:>5}  val_MAE {mae:.4f}"
                f"  best {self.best_mae:.4f}@{self.best_iter}  {elapsed(self.start)}"
            )

class XGBProgressCallback(xgb.callback.TrainingCallback):
    def __init__(self, fold, n_folds, total_rounds, log_every=200):
        self.fold = fold; self.n_folds = n_folds
        self.total = total_rounds; self.log_every = log_every
        self.start = time.time(); self.best_mae = float('inf'); self.best_iter = 0
    def after_iteration(self, model, epoch, evals_log):
        it = epoch + 1
        try: mae = list(evals_log.values())[0]['mae'][-1]
        except: return False
        if mae < self.best_mae: self.best_mae = mae; self.best_iter = it
        if it % self.log_every == 0:
            filled = int(30 * it / self.total)
            bar = '█'*filled + '░'*(30-filled)
            _print_progress(
                f"  Fold {self.fold}/{self.n_folds}  [{bar}] {it/self.total*100:5.1f}%"
                f"  iter {it:>5}  val_MAE {mae:.4f}"
                f"  best {self.best_mae:.4f}@{self.best_iter}  {elapsed(self.start)}"
            )
        return False

def cat_verbose_step(n_estimators):
    return max(1, n_estimators // 5)

# ============================================================
# 1. 데이터 로드
# ============================================================
path = r'C:\Users\82108\Downloads\open'

print("▶ 데이터 로드 중...", end=' ', flush=True)
t0 = time.time()
train  = pd.read_csv(os.path.join(path, 'train.csv'))
test   = pd.read_csv(os.path.join(path, 'test.csv'))
layout = pd.read_csv(os.path.join(path, 'layout_info.csv'))
print(f"완료 ({elapsed(t0)})  train {len(train):,}행 / test {len(test):,}행")

TARGET  = 'avg_delay_minutes_next_30m'
ID_COLS = ['ID', 'layout_id', 'scenario_id']
N_FOLDS = 5

# ============================================================
# 2. 결측치 처리
# ============================================================
def handle_missing_values(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)
    cols_to_fix = [c for c in df.columns
                   if df[c].isnull().any() and c not in (ID_COLS + [TARGET])]
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].ffill()
    df[cols_to_fix] = df.groupby('scenario_id')[cols_to_fix].bfill()
    df[cols_to_fix] = df[cols_to_fix].fillna(df[cols_to_fix].median())
    return df

# ============================================================
# 3. 기본 피처 엔지니어링
# ============================================================
def add_features(df):
    df = df.sort_values(['scenario_id', 'ID']).reset_index(drop=True)

    df['timeslot']          = df.groupby('scenario_id').cumcount()
    df['robot_efficiency']  = df['robot_active'] / (df['robot_total'] + 1e-6)
    df['robot_density']     = df['robot_total']  / (df['floor_area_sqm'] + 1e-6)
    df['order_per_station'] = df['order_inflow_15m'] / (df['pack_station_count'] + 1e-6)
    df['robot_per_station'] = df['robot_active'] / (df['pack_station_count'] + 1e-6)
    df['cumulative_orders'] = df.groupby('scenario_id')['order_inflow_15m'].cumsum()
    df['order_pressure']    = df['cumulative_orders'] / (df['pack_station_count'] + 1e-6)

    if 'congestion_score' in df.columns:
        df['risk_index']  = df['congestion_score'] * (1 - df['robot_efficiency'])
        df['bottle_neck'] = df['order_per_station'] * df['congestion_score']
    if 'low_battery_ratio' in df.columns:
        df['battery_risk'] = df['low_battery_ratio'] * df['robot_total']
    if 'battery_mean' in df.columns and 'battery_std' in df.columns:
        df['battery_cv'] = df['battery_std'] / (df['battery_mean'] + 1e-6)
    if 'avg_trip_distance' in df.columns:
        df['trip_per_robot']  = df['avg_trip_distance'] / (df['robot_active'] + 1e-6)
        df['trip_congestion'] = df['avg_trip_distance'] * df.get('congestion_score', 0)
    if 'pack_utilization' in df.columns:
        df['pack_order_ratio'] = df['pack_utilization'] / (df['order_per_station'] + 1e-6)

    roll_cols = ['order_per_station', 'congestion_score',
                 'pack_utilization', 'avg_trip_distance', 'low_battery_ratio']
    for col in roll_cols:
        if col not in df.columns: continue
        grp = df.groupby('scenario_id')[col]
        df[f'{col}_roll3_mean'] = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
        df[f'{col}_roll5_mean'] = grp.transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
        df[f'{col}_roll3_std']  = grp.transform(lambda x: x.shift(1).rolling(3, min_periods=1).std().fillna(0))

    for lag in [1, 2]:
        if 'order_per_station' in df.columns:
            df[f'order_per_station_lag{lag}'] = (
                df.groupby('scenario_id')['order_per_station'].shift(lag).bfill())

    scen_agg_cols = [
        'congestion_score', 'order_inflow_15m', 'battery_mean',
        'pack_utilization', 'avg_trip_distance', 'low_battery_ratio',
        'max_zone_density', 'sku_concentration', 'robot_idle',
        'outbound_truck_wait_min', 'order_per_station', 'robot_efficiency',
        'order_pressure', 'risk_index', 'battery_risk', 'battery_cv',
    ]
    for col in scen_agg_cols:
        if col not in df.columns: continue
        stats = df.groupby('scenario_id')[col].agg(['mean', 'max', 'min', 'std']).reset_index()
        stats.columns = ['scenario_id'] + [f'{col}_scen_{f}' for f in ['mean', 'max', 'min', 'std']]
        df = df.merge(stats, on='scenario_id', how='left')

    if 'congestion_score_scen_mean' in df.columns and 'pack_utilization_scen_mean' in df.columns:
        df['cong_pack_interaction']   = df['congestion_score_scen_mean'] * df['pack_utilization_scen_mean']
    if 'avg_trip_distance_scen_mean' in df.columns and 'congestion_score_scen_mean' in df.columns:
        df['trip_cong_interaction']   = df['avg_trip_distance_scen_mean'] * df['congestion_score_scen_mean']
    if 'low_battery_ratio_scen_mean' in df.columns and 'robot_efficiency_scen_mean' in df.columns:
        df['battery_efficiency_risk'] = df['low_battery_ratio_scen_mean'] * (1 - df['robot_efficiency_scen_mean'])

    # 시나리오 내 상대위치 피처
    for col in ['congestion_score', 'order_per_station', 'pack_utilization', 'avg_trip_distance']:
        scen_mean_col = f'{col}_scen_mean'
        if col in df.columns and scen_mean_col in df.columns:
            df[f'{col}_rel_to_scen'] = df[col] / (df[scen_mean_col] + 1e-6)

    # 시나리오 내 백분위 순위
    for col in ['congestion_score', 'order_per_station', 'pack_utilization']:
        if col not in df.columns: continue
        df[f'{col}_scen_rank'] = df.groupby('scenario_id')[col].rank(pct=True)

    return df

def preprocess_all(df, layout_df):
    df = df.merge(layout_df, on='layout_id', how='left')
    df = handle_missing_values(df)
    df = add_features(df)
    df['layout_type'] = pd.factorize(df['layout_type'])[0]
    return df

print("▶ 전처리 중...", end=' ', flush=True)
t0 = time.time()
train = preprocess_all(train, layout)
test  = preprocess_all(test,  layout)
print(f"완료 ({elapsed(t0)})")

# ============================================================
# 4. 타겟 인코딩
# ============================================================
print("▶ 타겟 인코딩 중...", end=' ', flush=True)
t0 = time.time()

TE_COLS   = [c for c in ['layout_id', 'timeslot', 'layout_type', 'shift_hour', 'day_of_week']
             if c in train.columns]
SMOOTHING = 20
kf_te     = GroupKFold(n_splits=N_FOLDS)
groups_te = train['scenario_id']

for col in TE_COLS:
    te_col = f'{col}_te'
    train[te_col] = np.nan
    global_mean   = train[TARGET].mean()
    for tr_idx, val_idx in kf_te.split(train, train[TARGET], groups=groups_te):
        tr_df  = train.iloc[tr_idx]
        stats  = tr_df.groupby(col)[TARGET].agg(['mean', 'count'])
        smooth = (stats['count'] * stats['mean'] + SMOOTHING * global_mean) / (stats['count'] + SMOOTHING)
        train.loc[val_idx, te_col] = train.iloc[val_idx][col].map(smooth).fillna(global_mean)
    stats_full  = train.groupby(col)[TARGET].agg(['mean', 'count'])
    smooth_full = (stats_full['count'] * stats_full['mean'] + SMOOTHING * global_mean) / (stats_full['count'] + SMOOTHING)
    test[te_col] = test[col].map(smooth_full).fillna(global_mean)

print(f"완료 ({elapsed(t0)})")

# ============================================================
# 5. 타겟 Lag 피처
# ============================================================
print("▶ 타겟 Lag 피처 생성 중...", end=' ', flush=True)
t0 = time.time()

TARGET_LAG_COLS = [f'target_lag{lag}' for lag in [1, 2, 3]]
train_global_mean = train[TARGET].mean()

for lag in [1, 2, 3]:
    col_name = f'target_lag{lag}'
    train[col_name] = (
        train.sort_values(['scenario_id', 'ID'])
             .groupby('scenario_id')[TARGET].shift(lag)
    )
    scen_mean = train.groupby('scenario_id')[TARGET].transform('mean')
    train[col_name] = train[col_name].fillna(scen_mean)
    test[col_name]  = train_global_mean  # 초기값, 순차예측에서 덮어씀

print(f"완료 ({elapsed(t0)})")

feature_cols = [c for c in train.columns if c not in ID_COLS + [TARGET]]
print(f"▶ 총 피처 수: {len(feature_cols)}개")

y      = np.log1p(train[TARGET])
groups = train['scenario_id']
kf     = GroupKFold(n_splits=N_FOLDS)

oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

lgb_models, xgb_models, cat_models = [], [], []

# ============================================================
# 6. LightGBM
# ============================================================
section("LightGBM 학습")
lgb_params = dict(
    n_estimators=10000, learning_rate=0.005,
    max_depth=8, num_leaves=127, min_child_samples=30,
    subsample=0.75, subsample_freq=1, colsample_bytree=0.6,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42, verbose=-1,
)
lgb_fold_maes = []
lgb_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    X_tr, y_tr   = train.loc[tr_idx, feature_cols], y.iloc[tr_idx]
    X_val, y_val = train.loc[val_idx, feature_cols], y.iloc[val_idx]
    cb_p  = LGBProgressCallback(fold, N_FOLDS, lgb_params['n_estimators'])
    model = lgb.LGBMRegressor(**lgb_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], eval_metric='mae',
              callbacks=[lgb.early_stopping(400, verbose=False), lgb.log_evaluation(-1), cb_p])
    fold_pred = np.expm1(model.predict(X_val))
    oof_lgb[val_idx] = fold_pred
    lgb_models.append(model)
    fold_mae = mean_absolute_error(train.loc[val_idx, TARGET], fold_pred)
    lgb_fold_maes.append(fold_mae)
    print(f"\n  ✔ Fold {fold}  MAE {fold_mae:.4f}  best iter {model.best_iteration_:,}  {elapsed(lgb_start)}")

lgb_mae = mean_absolute_error(train[TARGET], oof_lgb)
print(f"\n  ▶ LightGBM OOF MAE : {lgb_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in lgb_fold_maes]}")

# ============================================================
# 7. XGBoost
# ============================================================
section("XGBoost 학습")
xgb_params = dict(
    n_estimators=10000, learning_rate=0.005, max_depth=7,
    subsample=0.75, colsample_bytree=0.6,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42,
    tree_method='hist', early_stopping_rounds=400,
    eval_metric='mae', verbosity=0,
)
xgb_fold_maes = []
xgb_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    X_tr, y_tr   = train.loc[tr_idx, feature_cols], y.iloc[tr_idx]
    X_val, y_val = train.loc[val_idx, feature_cols], y.iloc[val_idx]
    cb_p  = XGBProgressCallback(fold, N_FOLDS, 10000)
    model = xgb.XGBRegressor(**xgb_params, callbacks=[cb_p])
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    fold_pred = np.expm1(model.predict(X_val))
    oof_xgb[val_idx] = fold_pred
    xgb_models.append(model)
    fold_mae = mean_absolute_error(train.loc[val_idx, TARGET], fold_pred)
    xgb_fold_maes.append(fold_mae)
    print(f"\n  ✔ Fold {fold}  MAE {fold_mae:.4f}  best iter {model.best_iteration:,}  {elapsed(xgb_start)}")

xgb_mae = mean_absolute_error(train[TARGET], oof_xgb)
print(f"\n  ▶ XGBoost OOF MAE  : {xgb_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in xgb_fold_maes]}")

# ============================================================
# 8. CatBoost
# ============================================================
section("CatBoost 학습")
cat_params = dict(
    iterations=10000, 
    learning_rate=0.005,
    depth=8, 
    l2_leaf_reg=3.0,
    bootstrap_type='MVS',    # 이 줄을 추가했습니다.
    subsample=0.75,          # 이제 에러 없이 작동합니다.
    colsample_bylevel=0.6,
    loss_function='MAE', 
    eval_metric='MAE',
    random_seed=42, 
    task_type='CPU',         # 코랩 GPU 환경 활용
    early_stopping_rounds=400,
)
cat_fold_maes = []
cat_start = time.time()

for fold, (tr_idx, val_idx) in enumerate(kf.split(train, y, groups=groups), 1):
    X_tr, y_tr   = train.loc[tr_idx, feature_cols], y.iloc[tr_idx]
    X_val, y_val = train.loc[val_idx, feature_cols], y.iloc[val_idx]
    print(f"\n  Fold {fold}/{N_FOLDS} 학습 중...")
    model = cb.CatBoostRegressor(**cat_params)
    model.fit(X_tr, y_tr, eval_set=(X_val, y_val),
              verbose=cat_verbose_step(cat_params['iterations']),
              use_best_model=True)
    fold_pred = np.expm1(model.predict(X_val))
    oof_cat[val_idx] = fold_pred
    cat_models.append(model)
    fold_mae = mean_absolute_error(train.loc[val_idx, TARGET], fold_pred)
    cat_fold_maes.append(fold_mae)
    print(f"  ✔ Fold {fold}  MAE {fold_mae:.4f}  best iter {model.best_iteration_:,}  {elapsed(cat_start)}")

cat_mae = mean_absolute_error(train[TARGET], oof_cat)
print(f"\n  ▶ CatBoost OOF MAE : {cat_mae:.4f}")
print(f"    Fold별: {[f'{m:.4f}' for m in cat_fold_maes]}")

# ============================================================
# ★ 9. Test 순차 예측 — 타임슬롯 단위 배치 처리 (핵심 최적화)
# ============================================================
# 이전 버전: 행 1개씩 predict → 750,000번 호출 (매우 느림)
# 이번 버전: 같은 타임슬롯의 모든 시나리오를 한번에 배치 predict
#            → 25번 호출로 끝 (약 100배 빠름)
# ============================================================
section("Test 순차 예측 (배치 최적화)")

w_lgb = 1 / lgb_mae
w_xgb = 1 / xgb_mae
w_cat = 1 / cat_mae
w_sum = w_lgb + w_xgb + w_cat

def ensemble_predict(X, lgb_models, xgb_models, cat_models, w_lgb, w_xgb, w_cat, w_sum):
    """5개 fold 모델 평균 후 3-way 앙상블"""
    p_lgb = np.mean([np.expm1(m.predict(X)) for m in lgb_models], axis=0)
    p_xgb = np.mean([np.expm1(m.predict(X)) for m in xgb_models], axis=0)
    p_cat = np.mean([np.expm1(m.predict(X)) for m in cat_models], axis=0)
    return (w_lgb * p_lgb + w_xgb * p_xgb + w_cat * p_cat) / w_sum

test = test.sort_values(['scenario_id', 'ID']).reset_index(drop=True)

# 타임슬롯별로 그룹핑 (전체 시나리오에서 같은 슬롯 위치)
test['timeslot'] = test.groupby('scenario_id').cumcount()
max_slot = test['timeslot'].max()

test_preds = np.zeros(len(test))
seq_start  = time.time()

print(f"  타임슬롯 0~{max_slot} 순차 예측 중...")

for slot in range(max_slot + 1):
    slot_mask = test['timeslot'] == slot
    slot_idx  = test.index[slot_mask]

    # lag 피처 채우기 — 이전 슬롯 예측값 참조
    for lag in [1, 2, 3]:
        lag_col      = f'target_lag{lag}'
        prev_slot    = slot - lag
        if prev_slot >= 0:
            prev_mask = test['timeslot'] == prev_slot
            # 같은 시나리오의 이전 슬롯 예측값 매핑
            prev_df = test.loc[prev_mask, ['scenario_id']].copy()
            prev_df['pred'] = test_preds[prev_mask]
            scen_to_pred = prev_df.set_index('scenario_id')['pred'].to_dict()
            test.loc[slot_idx, lag_col] = test.loc[slot_idx, 'scenario_id'].map(scen_to_pred).fillna(train_global_mean)
        # prev_slot < 0이면 초기값(train_global_mean) 그대로 유지

    # 현재 슬롯 전체 시나리오 한번에 배치 예측
    X_slot = test.loc[slot_idx, feature_cols]
    test_preds[slot_idx] = ensemble_predict(
        X_slot, lgb_models, xgb_models, cat_models, w_lgb, w_xgb, w_cat, w_sum
    )

    if slot % 5 == 0 or slot == max_slot:
        print(f"  슬롯 {slot:>2}/{max_slot}  완료  {elapsed(seq_start)}", flush=True)

print(f"  ✔ 순차 예측 완료  ({elapsed(seq_start)})")

# ============================================================
# 10. 최종 앙상블 & 제출
# ============================================================
section("앙상블 & 제출")

oof_final = (w_lgb*oof_lgb + w_xgb*oof_xgb + w_cat*oof_cat) / w_sum
final_mae = mean_absolute_error(train[TARGET], oof_final)

print(f"  LightGBM {w_lgb/w_sum*100:.1f}%  (OOF MAE {lgb_mae:.4f})")
print(f"  XGBoost  {w_xgb/w_sum*100:.1f}%  (OOF MAE {xgb_mae:.4f})")
print(f"  CatBoost {w_cat/w_sum*100:.1f}%  (OOF MAE {cat_mae:.4f})")
print(f"\n  ★ 앙상블 최종 OOF MAE : {final_mae:.4f}")
print(f"  ★ 전체 소요 시간       : {elapsed(lgb_start)}")

submission = pd.DataFrame({'ID': test['ID'], TARGET: test_preds})
save_path  = os.path.join(path, 'submission_v6.csv')
submission.to_csv(save_path, index=False)
print(f"\n  저장 완료 → {save_path}")